# 13 — Pure vs Mixed

Fill the slot: build statistical mixtures of states, meet the maximally mixed state $I/2$, set it beside $|+\rangle$ — same diagonal, no coherence — find the measurement that tells them apart, learn to measure "how mixed" with purity, and hold the single contrast the whole course is built to explain.

**Prerequisites:** `01/12_from_states_to_density_matrix` (outer product, $\mathrm{tr}(\rho A)$, $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$, the three conditions, section 5), `01/08_hadamard_and_measurement_bases` (measuring in the $X$ basis), `01/11_shot_noise_and_first_circuit` (§5: a Bloch vector shorter than $1$).

**What you'll be able to do**
- Distinguish a **superposition** from a **statistical mixture** — two different kinds of uncertainty — and name the measurement that separates them.
- Build $\rho = \sum_i p_i |\psi_i\rangle\langle\psi_i|$ by hand, see that its Bloch vector is the *average* of the Bloch vectors (so it lives *inside* the sphere), and that $I/2$ is the centre.
- Derive **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2)$ and the eigenvalues $\tfrac12(1 \pm |\vec r|)$ of a qubit $\rho$; meet von Neumann entropy as a preview.
- State the seed of the course — $|+\rangle$ vs $I/2$ — precisely, including what it does and does not promise.

This is, quietly, the most important notebook in the foundations. You will build two states that look identical to a $Z$ measurement, yet are about as different as two states can be — and that difference is the exact reason the rest of the course exists. Take your time here; it rewards it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum import density
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, bloch_vector
from qot_course.quantum.gates import PAULI_X, PAULI_Y, PAULI_Z, HADAMARD, S_DAG, apply_gate

rng = np.random.default_rng(0)   # random states for the exercises
viz.use_course_style()

def outer(psi):
    """|psi><psi| by hand (01/12)."""
    return np.outer(psi, psi.conj())

## 1. Two kinds of uncertainty

$|+\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)$ is uncertain in a *quantum* way: a $Z$ measurement gives $0$ or $1$ with probability $\tfrac12$ each, and yet the state is perfectly definite — `01/08` showed that an $X$ measurement returns $+$ *every single time*. Nothing is unknown; the randomness is in the measurement, not in your knowledge.

Now the other kind. A friend flips a fair coin and hands you $|0\rangle$ on heads, $|1\rangle$ on tails, without telling you which. The qubit is in one definite state; *you* do not know which. `01/12` §5 told you how to write that: every prediction is $\mathrm{tr}(\rho A)$, linear in $\rho$, so your best description is the **average density matrix**

$$ \rho_{\text{mix}} = \tfrac12\,|0\rangle\langle 0| + \tfrac12\,|1\rangle\langle 1| = \tfrac12\begin{pmatrix}1&0\\0&0\end{pmatrix} + \tfrac12\begin{pmatrix}0&0\\0&1\end{pmatrix} = \tfrac12\begin{pmatrix}1&0\\0&1\end{pmatrix} = \frac{I}{2}, $$

the **maximally mixed state**. In general a **statistical mixture** of states $|\psi_i\rangle$ held with probabilities $p_i$ is $\rho = \sum_i p_i\,|\psi_i\rangle\langle\psi_i|$ — a weighted sum of outer products, Hermitian, unit trace and positive because each term is and the weights are non-negative. Build $I/2$ by hand, compare it with $|+\rangle$, and then run the experiment that separates them.

In [ ]:
rho_plus = outer(KET_PLUS)
rho_mix = 0.5 * outer(KET_0) + 0.5 * outer(KET_1)
print("rho_mix = 1/2 |0><0| + 1/2 |1><1| =\n", np.round(rho_mix, 3))
print("equals I/2:", np.allclose(rho_mix, np.eye(2) / 2), "   library maximally_mixed / mixed_state agree:",
      np.allclose(rho_mix, density.maximally_mixed(2)), np.allclose(rho_mix, density.mixed_state([KET_0, KET_1], [0.5, 0.5])))
print("valid density matrix:", density.is_density_matrix(rho_mix), "   projector (rho@rho == rho)?", np.allclose(rho_mix @ rho_mix, rho_mix), "  -- rho@rho =\n", np.round(rho_mix @ rho_mix, 3))
# linearity (01/12 section 3): the mixture's prediction is the average of the parts' predictions
print("tr(rho_mix Z) =", np.trace(rho_mix @ PAULI_Z).real, " = 1/2 <0|Z|0> + 1/2 <1|Z|1> =", 0.5 * np.vdot(KET_0, PAULI_Z @ KET_0).real + 0.5 * np.vdot(KET_1, PAULI_Z @ KET_1).real)

print("\n            diagonal (Z probabilities)   coherence |rho_01|")
for name, r in [("|+>", rho_plus), ("I/2", rho_mix)]:
    print(f"{name:10s}  {np.round(np.diag(r).real, 3)}                 {abs(r[0, 1]):.3f}")

# the discriminating experiment: measure in the X basis, P(+-) = <+-|rho|+->  (01/12 section 3)
print("\nX-basis probabilities  P(+), P(-) = <+|rho|+>, <-|rho|->")
for name, r in [("|+>", rho_plus), ("I/2", rho_mix)]:
    p_plus = np.vdot(KET_PLUS, r @ KET_PLUS).real
    p_minus = np.vdot(KET_MINUS, r @ KET_MINUS).real
    print(f"{name:10s}  P(+) = {p_plus:.3f}   P(-) = {p_minus:.3f}")

viz.plot_density_matrix(rho_plus, title="|+> : a pure superposition", basis_labels=["|0>", "|1>"])
viz.plot_density_matrix(rho_mix, title="I/2 : a classical mixture (maximally mixed)", basis_labels=["|0>", "|1>"])
plt.show()

**Read the output and the figures.** The hand sum of two outer products is $I/2$, a valid density matrix — and *not* a projector: $\rho^2 \ne \rho$, so it is no pure state's $|\psi\rangle\langle\psi|$, exactly as `01/12` §5 promised. Look at the two heatmaps together. $|+\rangle$ and $I/2$ have the **same diagonal** $(0.5, 0.5)$: a $Z$ measurement returns $50/50$ for both and cannot tell them apart. But $|+\rangle$ carries **coherence** $\tfrac12$ in the corners and $I/2$ has none — and the $X$ measurement sees it: $|+\rangle$ gives $+$ with certainty, $I/2$ gives $+$ and $-$ fifty-fifty. That is the whole difference between "genuinely both at once" and "one of the two, unknown", made operational: a superposition interferes (`01/02`), a mixture does not. The "with some probability" of `01/11` finally has a home — in the weights $p_i$, not in the amplitudes.

## 2. Where mixtures live: inside the sphere

`01/12` earned $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ with $\vec r = (\mathrm{tr}\rho X, \mathrm{tr}\rho Y, \mathrm{tr}\rho Z)$, for a pure state. The formula never used purity: it only used that the Paulis are a basis. So it holds for *any* $\rho$ — and since the trace is linear, the Bloch vector of a mixture is the **weighted average of the Bloch vectors** of its parts:

$$ \vec r\big(\textstyle\sum_i p_i \rho_i\big) = \sum_i p_i\,\vec r(\rho_i). $$

An average of unit vectors has length *at most* $1$, with equality only when all the vectors agree. So a genuine mixture sits strictly **inside** the Bloch sphere — and $I/2$ — here the average of two antipodes — sits at the very **centre**, $\vec r = 0$. This is what a Bloch vector shorter than $1$ means: a mixture. (In `01/11` §5 the shortfall you measured came from the *meter*, not the state; a real device's decay and gate errors produce the genuine article, and `01/15` reconstructs it.) (`01/14` turns the inside of the sphere into the **Bloch ball** and measures distances in it.)

Two more things follow by hand. A mixed state does **not remember how it was made**: $\tfrac12|+\rangle\langle+| + \tfrac12|-\rangle\langle-| = \tfrac12\cdot\tfrac12\begin{pmatrix}1&1\\1&1\end{pmatrix} + \tfrac12\cdot\tfrac12\begin{pmatrix}1&-1\\-1&1\end{pmatrix} = I/2$ as well — a coin between $|+\rangle$ and $|-\rangle$, or between $|{+i}\rangle$ and $|{-i}\rangle$, gives *the same* $\rho$ as a coin between $|0\rangle$ and $|1\rangle$, and no experiment can tell the three preparations apart. And a $\rho$ that is not a projector is a mixture of *something*: its eigen-decomposition $\rho = \sum_k \lambda_k |e_k\rangle\langle e_k|$ (eigenvalues $\lambda_k \ge 0$, summing to $1$ — the three conditions of `01/12` §4 again) is one way to prepare it, with the eigenvalues as the coin. Let's check all of this with numbers.

In [ ]:
def bloch_of(rho):
    """Bloch vector of any qubit rho: (tr rho X, tr rho Y, tr rho Z)  (01/12 section 3)."""
    return np.array([np.trace(rho @ P).real for P in (PAULI_X, PAULI_Y, PAULI_Z)])

# the Bloch vector of a mixture is the average of the Bloch vectors
psi_a, psi_b = qubit_state(np.pi / 3, np.pi / 5), qubit_state(2 * np.pi / 3, -np.pi / 2)
p = 0.7
rho_ab = p * outer(psi_a) + (1 - p) * outer(psi_b)
print("0.7 * r(a) + 0.3 * r(b) =", np.round(p * bloch_vector(psi_a) + (1 - p) * bloch_vector(psi_b), 4))
print("r(0.7 rho_a + 0.3 rho_b) =", np.round(bloch_of(rho_ab), 4), "   length", round(float(np.linalg.norm(bloch_of(rho_ab))), 4), "< 1")
print("(I + r.sigma)/2 rebuilds rho_ab:", np.allclose(density.density_from_bloch(bloch_of(rho_ab)), rho_ab), "  (01/12's formula, now for a mixture)")
print("\nr(|+>) =", np.round(bloch_of(rho_plus), 3), "  length 1 (on the sphere);   r(I/2) =", np.round(bloch_of(rho_mix), 3), "  length 0 (the centre)")

# a mixed state does not remember how it was made
KET_PLUS_I, KET_MINUS_I = qubit_state(np.pi / 2, np.pi / 2), qubit_state(np.pi / 2, -np.pi / 2)
from_pm = 0.5 * outer(KET_PLUS) + 0.5 * outer(KET_MINUS)
from_pmi = 0.5 * outer(KET_PLUS_I) + 0.5 * outer(KET_MINUS_I)
print("\n1/2|+><+| + 1/2|-><-| == I/2 :", np.allclose(from_pm, rho_mix), "    1/2|+i><+i| + 1/2|-i><-i| == I/2 :", np.allclose(from_pmi, rho_mix))

# every rho is a mixture of its eigenvectors, with the eigenvalues as the coin
lam, vecs = np.linalg.eigh(rho_ab)
print("\neigenvalues of rho_ab:", np.round(lam, 4), " (non-negative, sum", round(lam.sum(), 4), ")")
print("sum_k lambda_k |e_k><e_k| == rho_ab:", np.allclose(sum(l * outer(vecs[:, k]) for k, l in enumerate(lam)), rho_ab))
print("eigenvalues vs (1 -+ |r|)/2:", np.round([(1 - np.linalg.norm(bloch_of(rho_ab))) / 2, (1 + np.linalg.norm(bloch_of(rho_ab))) / 2], 4))

**Read the output.** The Bloch vector of the $0.7/0.3$ mixture is the $0.7/0.3$ average of the two Bloch vectors, its length is below $1$, and $\tfrac12(I + \vec r\cdot\vec\sigma)$ rebuilds the mixture exactly — `01/12`'s packing formula was never about purity. $|+\rangle$ sits on the sphere, $I/2$ at its centre. Three different coins — $|0\rangle/|1\rangle$, $|+\rangle/|-\rangle$, $|{+i}\rangle/|{-i}\rangle$ — produce the identical $I/2$: the density matrix keeps only what is observable, and "how it was prepared" is not. And any $\rho$ is a mixture of its own eigenvectors, weighted by its eigenvalues, which for a qubit are $\tfrac12(1 \pm |\vec r|)$ — section 3 derives that last formula in one line, and exercise 4 uses it.

## 3. Measuring mixedness: purity

Two numbers make "how mixed" precise. The first, **purity** $\mathrm{tr}(\rho^2)$, you can derive right now from `01/12`'s Pauli toolkit. With $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$,

$$ \rho^2 = \tfrac14\big(I + 2\,\vec r\cdot\vec\sigma + (\vec r\cdot\vec\sigma)^2\big), \qquad (\vec r\cdot\vec\sigma)^2 = |\vec r|^2\, I $$

(the last step is `01/10` exercise 4: a "Pauli along $\vec n$" squares to $I$ because $X^2 = Y^2 = Z^2 = I$ and different Paulis anticommute). The same fact gives the eigenvalues promised in section 2: $\vec r\cdot\vec\sigma$ squares to $|\vec r|^2 I$ and is traceless, so its eigenvalues are $\pm|\vec r|$, and those of $\rho$ are $\tfrac12(1 \pm |\vec r|)$. Taking the trace, $\mathrm{tr}\,I = 2$ and $\mathrm{tr}\,\sigma_i = 0$, so

$$ \mathrm{tr}(\rho^2) = \tfrac12\big(1 + |\vec r|^2\big) . $$

Purity is $1$ exactly on the sphere ($|\vec r| = 1$: pure, $\rho^2 = \rho$) and falls to its minimum $\tfrac12$ at the centre ($I/2$); it is a direct reading of how far inside the sphere the state sits. In the eigenbasis of section 2, $\mathrm{tr}(\rho^2) = \lambda_0^2 + \lambda_1^2$ — the probability that two draws of the preparing coin agree, which is why it is $1$ only when the coin is rigged.

The second number is the **von Neumann entropy** $S(\rho) = -\mathrm{tr}(\rho\log_2\rho) = -\sum_k \lambda_k\log_2\lambda_k$ (base-$2$ logarithms: bits): the entropy of the eigenvalue coin — $0$ for a pure state, $1$ bit for $I/2$. Take it as a preview today: `02/01` defines entropy properly (it is Shannon's entropy of the eigenvalues), and `02/09` makes $S$ the root of the quantum information measures. Let's compute both along the straight line from $|+\rangle$ to $I/2$ — the coherence draining out of the corners.

In [ ]:
ts = np.linspace(0, 1, 101)
rhos_t = [(1 - t) * rho_plus + t * rho_mix for t in ts]
coh = np.array([abs(r[0, 1]) for r in rhos_t])
r_len = np.array([np.linalg.norm(bloch_of(r)) for r in rhos_t])
pur = np.array([np.trace(r @ r).real for r in rhos_t])
ent = np.array([density.von_neumann_entropy(r) for r in rhos_t])

print("purity by hand (1 + |r|^2)/2 == tr(rho^2) along the line:", np.allclose(pur, (1 + r_len**2) / 2), "   library density.purity agrees:", np.allclose(pur, [density.purity(r) for r in rhos_t]))
print("\n  t     coherence   |r|     purity   entropy (bits)")
for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    i = int(round(t * 100))
    print(f"{t:4.2f}    {coh[i]:.3f}      {r_len[i]:.3f}   {pur[i]:.3f}    {ent[i]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.plot(ts, coh, color=COLORS["highlight"], lw=2, label=r"coherence $|\rho_{01}|$")
ax.plot(ts, r_len, color=COLORS["quantum"], lw=2, ls="--", label=r"Bloch length $|\vec r|$")
ax.plot(ts, pur, color=COLORS["flow"], lw=2, label=r"purity tr$(\rho^2) = (1+|\vec r|^2)/2$")
ax.set_xlabel(r"$t$:  $\rho(t) = (1-t)\,|+\rangle\langle+| + t\,I/2$"); ax.set_ylim(0, 1.05)
ax.set_title("From |+> to I/2: the coherence drains,\nthe state sinks toward the centre", pad=12)
ax.legend(fontsize=8, loc="lower left")
ax = axes[1]
ax.plot(ts, ent, color=COLORS["source"], lw=2, label=r"von Neumann entropy $S(\rho)$ (bits)")
ax.plot(ts, 1 - pur, color=COLORS["flow"], lw=2, ls=":", label=r"$1 - $ purity, for comparison")
ax.set_xlabel(r"$t$"); ax.set_ylim(0, 1.05)
ax.set_title("Entropy of the eigenvalue coin:\n0 bit (pure) to 1 bit (I/2)", pad=12)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

**Read the output and the figures.** The hand formula $\tfrac12(1 + |\vec r|^2)$ matches $\mathrm{tr}(\rho^2)$ at every point of the line (and the library's `purity`). Along the straight line the coherence falls linearly from $\tfrac12$ to $0$ and so does the Bloch length — here $|\vec r| = 2|\rho_{01}| = 1 - t$, because the diagonal never moves — while purity falls *quadratically*, from $1$ to $\tfrac12$: halfway along, the state still has purity $0.625$, not $0.75$ (exercise 3 asks you to predict this before checking). The entropy rises from $0$ to $1$ bit, steeply at first — the first drop of ignorance costs the most entropy — and rises to exactly one bit, where $1 - \mathrm{tr}\rho^2$ reaches only $\tfrac12$: the two measures agree on the endpoints and order states the same way along this line, but they are different rulers. Both report the same fact in two currencies: $|+\rangle$ is perfectly definite, $I/2$ carries a full bit of classical ignorance, and the diagonal never noticed.

## 4. The seed of the whole course

Sit with this for a moment. $|+\rangle$ and $I/2$ are indistinguishable to a $Z$ measurement — same diagonal — yet one holds maximal coherence and the other none; one is pure, the other maximally mixed; one lies on the sphere, the other at its centre.

A description that keeps the probabilities of **one** basis — the diagonal, a probability vector $(\tfrac12, \tfrac12)$, the same for both — cannot tell them apart; section 1 showed that the $X$ basis can, because the coherence *is* $\langle X\rangle/2$. A description that keeps the **whole operator** — every shadow at once — can. That gap is what the entire course is built to explore:

> **Classical optimal transport moves probability vectors — the diagonal. Quantum optimal transport moves density matrices — the whole $\rho$.**

Be precise about what this promises. It does *not* say that a quantum method will beat a classical one whenever there is coherence around; what the quantum formalism can add over a classical metric must come from the corners, and whether it *does*, for a given question, depends on what the embedding writes into those corners. Module 04 tests exactly this, honestly (`04/13`–`04/16`): the payoff is a unified geometry in which "diagonal only" and "whole $\rho$" are two settings of one dial. You are holding the seed of it right now, and you built it from a coin flip.

## Your turn

1. **Two mixtures by hand.** First close `01/12` §5: build $0.7\,|0\rangle\langle 0| + 0.3\,|1\rangle\langle 1|$ with `outer` and confirm it is the $\mathrm{diag}(0.7, 0.3)$ "slot" state — what are its Bloch vector, length and purity? Then build $\rho = \tfrac12|0\rangle\langle 0| + \tfrac12|+\rangle\langle+|$ by hand and with `density.mixed_state([KET_0, KET_PLUS], [0.5, 0.5])`. Is its diagonal $50/50$? Is it pure? Compute its Bloch vector as the average of $(0,0,1)$ and $(1,0,0)$, its length, and from that its purity *before* calling `density.purity`; then its entropy.
2. **Three coins, one state.** Prepare $I/2$ as a $\tfrac12/\tfrac12$ coin between $|{+i}\rangle$ and $|{-i}\rangle$ (build $|{+i}\rangle$ as `apply_gate(S_DAG.conj().T, apply_gate(HADAMARD, KET_0))`, the $S$-then-$H$ route of `01/08`), and as a three-way coin with weights $\tfrac13$ between three *equatorial* states at $\phi = 0, 2\pi/3, 4\pi/3$ (Bloch vectors $120^\circ$ apart, averaging to zero). Confirm with `np.allclose`. Then show that *no* weights make a coin between $|0\rangle$, $|+\rangle$ and $|{+i}\rangle$ equal to $I/2$ — why not? (Hint: their three Bloch vectors all point into the same octant.)
3. **Predict, then check.** Before computing, predict the purity of the state exactly halfway between $|+\rangle$ and $I/2$ from $\tfrac12(1 + |\vec r|^2)$, and the purity of the state halfway between $|0\rangle$ and $|1\rangle$. Then verify with `density.purity`. Which of the two midpoints is more mixed, and why does the *distance between the endpoints* matter?
4. **Harder: eigenvalues and the general two-state mixture.** Show by hand that a qubit $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ has eigenvalues $\tfrac12(1 \pm |\vec r|)$ (hint: $(\vec r\cdot\vec\sigma)^2 = |\vec r|^2 I$, so $\vec r\cdot\vec\sigma$ has eigenvalues $\pm|\vec r|$), hence $\mathrm{tr}(\rho^2) = \lambda_0^2 + \lambda_1^2$ and $S(\rho)$ is the binary entropy of $\tfrac12(1 + |\vec r|)$. Then, for $\rho = p\,|\psi\rangle\langle\psi| + (1-p)\,|\varphi\rangle\langle\varphi|$, show $|\vec r|^2 = 1 - 2p(1-p)\big(1 - \vec r_\psi\cdot\vec r_\varphi\big)$ and conclude that the mixture is pure only if $p \in \{0, 1\}$ or $|\psi\rangle = |\varphi\rangle$ up to phase. Verify both formulas on 500 random $(p, |\psi\rangle, |\varphi\rangle)$ drawn with `rng`.

## What you built

- You separated superposition (randomness in the measurement, state perfectly definite) from a statistical mixture (randomness in your knowledge), built $I/2 = \tfrac12|0\rangle\langle0| + \tfrac12|1\rangle\langle1|$ by hand, and ran the $X$ measurement that tells it from $|+\rangle$.
- You showed that the Bloch vector of a mixture is the average of its parts' — strictly inside the sphere, $I/2$ at the centre — that a mixed state does not remember its preparation, and that every $\rho$ is a mixture of its eigenvectors.
- You derived purity $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2)$, watched it fall from $1$ to $\tfrac12$ along the line from $|+\rangle$ to $I/2$, and met von Neumann entropy as the entropy of the eigenvalue coin.
- You stated the seed of the course — diagonal versus whole $\rho$ — with its honest boundary.

You found the seed: the off-diagonal coherence that a classical, diagonal-only view throws away. Hold onto it — when module 04 asks what quantum optimal transport can add over a classical metric, *this* is the place it must come from. A genuinely big idea, and you built it from a coin flip.

**Dictionary so far** (the table from `01/12`, plus the row you earned today):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| marginal | partial trace |
| transport plan | quantum coupling |

(The last two rows are still forward-looking — earned in `01/16` and `04/04`.)

## What's next

Purity tells pure from mixed — but how far apart are *two* states, mixed or not? In `01/14_fidelity_and_trace_distance` we turn the inside of the sphere into the **Bloch ball**, pick up two honest yardsticks — fidelity, which extends the inner product of `01/05` to mixed states, and trace distance — and watch mixedness become a literal position in the ball.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §2.4.1–2.4.2 (ensembles of quantum states, the Bloch-ball picture, the "unitary freedom in the ensemble" — why a mixed state does not remember its preparation), §11.3 (von Neumann entropy), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- J. von Neumann, *Thermodynamik quantenmechanischer Gesamtheiten*, Gött. Nachr. 273–291 (1927) — the entropy of a density operator; book form *Mathematische Grundlagen der Quantenmechanik*, Springer (1932), ch. V. doi:10.1007/978-3-642-61409-5 (reprint)
- J. Preskill, *Lecture Notes for Physics 219: Quantum Computation*, ch. 2 (the density operator, the Bloch ball, ensembles) — https://www.preskill.caltech.edu/ph229/

**Previous:** `notebooks/01_QuantumFoundations/12_from_states_to_density_matrix.ipynb` · **Next:** `notebooks/01_QuantumFoundations/14_fidelity_and_trace_distance.ipynb`